In [ ]:
!pip install -q gradio groq langchain langchain-community langchain-groq langchain-huggingface langchain-text-splitters faiss-cpu sentence-transformers pypdf

In [ ]:
import os
from google.colab import userdata
os.environ["GROQ_API_KEY"] = userdata.get("GROQ_API_KEY")

print("key is set")

In [ ]:
import gradio as gr
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_groq import ChatGroq
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from langchain_huggingface import HuggingFaceEmbeddings


llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0)


prompt = ChatPromptTemplate.from_template(
    """Answer the question using only the resume context below.
If the answer is not in the context, say you cannot find it in the resume.

Context:
{context}

Question: {question}

Answer:"""
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = None
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")

def build_chain(pdf_path):
  global rag_chain
  pages = PyPDFLoader(pdf_path).load()
  splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
  chunks = splitter.split_documents(pages)
  store = FAISS.from_documents(chunks, embeddings)
  retriever = store.as_retriever(search_kwargs={"k":3})

  rag_chain = (
      {"context": retriever | format_docs, "question": RunnablePassthrough()}
      | prompt
      | llm
      | StrOutputParser()
  )
  return f"Resume indexed into {len(chunks)} chunks. Ask a question below."

def answer_question(question):
  if rag_chain is None:
    return "Please upload a resume first."
  return rag_chain.invoke(question)

with gr.Blocks(title="Resume Q&A bot") as demo:
     gr.Markdown("## Resume Q&A Bot\nUpload a resume PDF, then ask questions about it.")
     pdf = gr.File(label="Resume PDF", file_types=[".pdf"], type="filepath")
     status = gr.Textbox(label="Status", interactive=False)
     pdf.upload(build_chain, inputs=pdf, outputs=status)
     question = gr.Textbox(label="Question")
     answer = gr.Textbox(label="Answer", lines=2, max_lines=20)
     question.submit(answer_question, inputs=question, outputs=answer)

demo.launch(debug=True)